# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*I will use a transparent refresh-priority rule that combines four observable signals: search visibility, freshness risk, position opportunity, and content depth gap. The score ranks content for human review rather than making an automatic publishing decision. Reason codes will explain common review opportunities such as stale visible pages, declining pages with demand, thin visible pages, page-one decay risk, low CTR on visible pages, and low engagement on visible pages.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s):
    s = s.astype(float)
    minimum = s.min()
    maximum = s.max()
    if maximum == minimum:
        return pd.Series(0.0, index=s.index)
    return (s - minimum) / (maximum - minimum)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Baseline rule: visibility + freshness + position opportunity + depth gap")

Rows: 30000
Columns: 44
Baseline rule: visibility + freshness + position opportunity + depth gap


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*I will calculate a 0-to-1 baseline refresh score for every content item, rank all items from highest to lowest priority, assign a suggested action, and attach reason codes. The output will be saved as work/outputs/baseline_action_score.csv for later comparison with the ML model.

In [ ]:
# 1. Visibility: higher impressions = higher visibility
df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

# 2. Freshness risk: older updates = higher refresh risk
df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

# 3. Position opportunity:
# Pages with better positions and meaningful visibility get more opportunity score
df["position_opportunity_score"] = (
    (1 - normalize(
        df["avg_position"].clip(lower=1, upper=50)
    ))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

# 4. Depth gap:
# Shorter content with meaningful visibility gets more review priority
df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)

# Final transparent baseline score
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)


# Reason codes
def reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
            or
            (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
        )
    ):
        reasons.append("low_engagement_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(reason_codes, axis=1)


# Suggested action
def suggested_action(row):
    reasons = set(str(row["reason_codes"]).split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if (
        "stale_visible_page" in reasons
        or "declining_with_demand" in reasons
    ):
        return "refresh"

    return "monitor"


df["suggested_action"] = df.apply(
    suggested_action,
    axis=1
)


# Rank everything
df["baseline_rank"] = (
    df["baseline_refresh_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Sort
df = df.sort_values("baseline_rank")


# Required output
output_columns = [
    "content_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

output = df[output_columns]

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows scored:", len(output))
print("Top score:", round(output["baseline_refresh_score"].max(), 3))
print("Median score:", round(output["baseline_refresh_score"].median(), 3))

print("\nTop 20:")
display(output.head(20))

Saved: work/outputs/baseline_action_score.csv
Rows scored: 30000
Top score: 0.948
Median score: 0.446

Top 20:


,content_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count
21565,content_9532f197bbc8,1,0.947603,0.999633,0.8432,0.979233,0.999633,declining_with_demand|page_one_decay_risk|low_...,refresh,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,NaN
4644,content_4d1fe5b32dc2,2,0.941268,0.994167,0.8432,0.963733,0.994167,page_one_decay_risk|low_engagement_visible_page,monitor,97999,512,549,2.5,0.52,7.47,13.15,329,104,NaN
18954,content_07f2e7a6f38a,3,0.940461,0.994467,0.8432,0.959965,0.994467,page_one_decay_risk|low_engagement_visible_page,monitor,101078,856,780,2.7,0.85,2.05,4.60,313,104,NaN
17400,content_e5ae436f9a16,4,0.939997,0.996000,0.8432,0.955347,0.996000,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,117741,533,522,3.0,0.45,7.09,12.60,421,104,NaN
9348,content_3430a8b94511,5,0.939963,0.998167,0.8432,0.951314,0.998167,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,152617,440,534,3.3,0.29,6.18,11.04,329,104,NaN
25409,content_cbd93118300b,6,0.939665,0.997733,0.8432,0.950901,0.997733,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,145292,662,535,3.3,0.46,1.87,5.38,313,104,NaN
18458,content_9c195417f6ef,7,0.939353,0.991400,0.8432,0.961051,0.991400,page_one_decay_risk|low_engagement_visible_page,monitor,79146,574,515,2.5,0.73,1.55,2.79,313,104,NaN
13306,content_ba2acb4ebd04,8,0.938024,0.997567,0.8432,0.944635,0.997567,page_one_decay_risk|low_engagement_visible_page,monitor,142072,1185,1147,3.6,0.83,1.92,5.08,362,104,NaN
28354,content_79b25654070a,9,0.937766,0.997933,0.8432,0.942945,0.997933,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,148737,711,619,3.7,0.48,2.26,3.46,257,104,NaN
8275,content_adddad39251c,10,0.937520,0.996833,0.8432,0.943940,0.996833,page_one_decay_risk|low_engagement_visible_page,monitor,129239,711,688,3.6,0.55,3.92,6.32,329,104,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*I will review the top 20 ranked items as a human-review queue. Each item will have a suggested action, reason codes, and a confidence note. The ranking is directional, so a high score does not mean the item definitely needs a refresh. A recommendation could be wrong because of seasonality, low traffic volume, measurement changes, or context that is not represented in the available fields.

In [6]:
# ============================================================
# ML-08 — Random Forest vs Baseline
# Complete standalone cell
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Binary target
df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining:", df["is_declining"].sum())
print("Non-declining:", (df["is_declining"] == 0).sum())


# ------------------------------------------------------------
# 2. Client-holdout split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("\n--- Split ---")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print("Client overlap:", len(overlap))


# ============================================================
# 3. BASELINE
# ============================================================

baseline_test = test_df.copy()


# Visibility
visibility_raw = np.log1p(
    baseline_test["impressions_90d"]
)

baseline_test["visibility_score"] = (
    visibility_raw.rank(
        method="average",
        pct=True
    )
    .fillna(0)
)


# Freshness risk
baseline_test["freshness_risk_score"] = (
    baseline_test["days_since_last_update"]
    .rank(
        method="average",
        pct=True
    )
    .fillna(0)
)


# Position opportunity
position = (
    baseline_test["avg_position"]
    .astype(float)
    .clip(lower=1, upper=50)
)

position_min = position.min()
position_max = position.max()

if position_max == position_min:
    position_norm = pd.Series(
        0.0,
        index=position.index
    )
else:
    position_norm = (
        position - position_min
    ) / (
        position_max - position_min
    )

baseline_test["position_opportunity_score"] = (
    (1 - position_norm)
    * baseline_test["visibility_score"]
    * (baseline_test["avg_position"] > 0).astype(int)
)


# Content-depth gap
word_rank = (
    baseline_test["word_count"]
    .astype(float)
    .rank(
        method="average",
        pct=True
    )
    .fillna(0)
)

baseline_test["depth_gap_score"] = (
    (1 - word_rank)
    * baseline_test["visibility_score"]
)


# Final transparent baseline score
baseline_test["baseline_score"] = (
    0.40 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["freshness_risk_score"]
    + 0.25 * baseline_test["position_opportunity_score"]
    + 0.05 * baseline_test["depth_gap_score"]
)


# Rank baseline
baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_test["baseline_rank"] = np.arange(
    1,
    len(baseline_test) + 1
)


# Precision@50
k = 50

baseline_precision_at_50 = (
    baseline_test
    .head(k)["is_declining"]
    .sum()
    / k
)

print("\n--- BASELINE ---")
print(
    "Baseline Precision@50:",
    round(baseline_precision_at_50, 3)
)


# ============================================================
# 4. RANDOM FOREST
# ============================================================

candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

feature_cols = [
    c for c in candidate_features
    if c in df.columns
]

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]


# Fill missing values using TRAINING medians only
train_medians = X_train.median(
    numeric_only=True
)

X_train = (
    X_train
    .fillna(train_medians)
    .fillna(0)
)

X_test = (
    X_test
    .fillna(train_medians)
    .fillna(0)
)


# Train Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)


# Probability of declining
model_probability = (
    model.predict_proba(X_test)[:, 1]
)


# Model results
model_results = test_df[
    [
        "content_id",
        "client_id",
        "is_declining"
    ]
].copy()

model_results["model_score"] = (
    model_probability
)

model_results = model_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

model_results["model_rank"] = np.arange(
    1,
    len(model_results) + 1
)


# Model Precision@50
model_precision_at_50 = (
    model_results
    .head(k)["is_declining"]
    .sum()
    / k
)


# Recall
recall = recall_score(
    y_test,
    (model_probability >= 0.5).astype(int),
    zero_division=0
)


# ============================================================
# 5. COMPARISON
# ============================================================

difference = (
    model_precision_at_50
    - baseline_precision_at_50
)

print("\n--- MODEL ---")
print("Features used:", len(feature_cols))
print(
    "Model Precision@50:",
    round(model_precision_at_50, 3)
)
print(
    "Model Recall:",
    round(recall, 3)
)

print("\n--- COMPARISON ---")

comparison = pd.DataFrame({
    "method": [
        "Baseline",
        "Random Forest"
    ],
    "precision_at_50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

display(comparison)

print(
    "Precision@50 difference:",
    round(difference, 3)
)


# ============================================================
# 6. TOP 20 MODEL RESULTS
# ============================================================

print("\nTop 20 model recommendations:")

display(
    model_results.head(20)
)


# ============================================================
# 7. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop model features:")

display(
    importance.head(10)
)


# ============================================================
# 8. ERROR ANALYSIS
# ============================================================

model_results["predicted_declining"] = (
    model_results["model_score"] >= 0.5
).astype(int)

false_positives = model_results[
    (model_results["predicted_declining"] == 1)
    & (model_results["is_declining"] == 0)
]

false_negatives = model_results[
    (model_results["predicted_declining"] == 0)
    & (model_results["is_declining"] == 1)
]

print("\nFalse positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(10))

print("\nExample false negatives:")
display(false_negatives.head(10))

Rows: 30000
Columns: 45
Declining: 16262
Non-declining: 13738

--- Split ---
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0

--- BASELINE ---
Baseline Precision@50: 0.32

--- MODEL ---
Features used: 26
Model Precision@50: 1.0
Model Recall: 0.928

--- COMPARISON ---


,method,precision_at_50
0,Baseline,0.32
1,Random Forest,1.00


Precision@50 difference: 0.68

Top 20 model recommendations:


,content_id,client_id,is_declining,model_score,model_rank
0,content_bf313c86ff6b,client_f369cb89fc,1,0.990000,1
1,content_b23d650634c6,client_f369cb89fc,1,0.990000,2
2,content_0b50482209cd,client_bdd2d3af3a,1,0.990000,3
3,content_eb3b2c3bbc34,client_f369cb89fc,1,0.990000,4
4,content_d028d97a0f72,client_f369cb89fc,1,0.986667,5
5,content_f6bf66378677,client_f369cb89fc,1,0.986667,6
6,content_9234f5075e7a,client_f369cb89fc,1,0.986667,7
7,content_47806094bb4d,client_f369cb89fc,1,0.986667,8
8,content_9bbf99b4dc21,client_434c9b5ae5,1,0.983333,9
9,content_e3bf6539b4ba,client_bdd2d3af3a,1,0.983333,10



Top model features:


,feature,importance
19,impressions_prev_30d,0.217607
16,impressions_last_30d,0.187799
6,impressions_90d,0.083806
4,content_age_days,0.059346
23,avg_position,0.057861
14,days_with_impressions,0.055516
3,word_count,0.032660
18,sessions_last_30d,0.027692
22,ctr,0.024816
17,clicks_last_30d,0.024617



False positives: 345
False negatives: 226

Example false positives:


,content_id,client_id,is_declining,model_score,model_rank,predicted_declining
1310,content_816d77e36e14,client_8527a891e2,0,0.803333,1311,1
1323,content_41baf0722ad9,client_8527a891e2,0,0.800000,1324,1
1674,content_4d9f36001f06,client_8527a891e2,0,0.753333,1675,1
1682,content_8f1409b2674e,client_8527a891e2,0,0.750000,1683,1
1801,content_4dd569ee33c9,client_8527a891e2,0,0.733333,1802,1
1829,content_15e0e6081fe9,client_f369cb89fc,0,0.730000,1830,1
1838,content_94fa1e16f2b8,client_4e07408562,0,0.730000,1839,1
1844,content_e0666cdbf9c3,client_4e07408562,0,0.730000,1845,1
1871,content_09227e80fef5,client_8527a891e2,0,0.723333,1872,1
1881,content_5ce1a9d3e4d7,client_8527a891e2,0,0.723333,1882,1



Example false negatives:


,content_id,client_id,is_declining,model_score,model_rank,predicted_declining
3268,content_db3fd75bd4fe,client_f369cb89fc,1,0.496667,3269,0
3269,content_a4d677c4395d,client_e629fa6598,1,0.496667,3270,0
3270,content_557fe9494c53,client_4e07408562,1,0.496667,3271,0
3272,content_1180705d05bb,client_4e07408562,1,0.496667,3273,0
3275,content_2d6c50388f53,client_4e07408562,1,0.496667,3276,0
3276,content_71c7b5968117,client_4e07408562,1,0.496667,3277,0
3278,content_b2cb21940b47,client_f369cb89fc,1,0.496667,3279,0
3281,content_6d772c85e856,client_e629fa6598,1,0.496667,3282,0
3282,content_ec5ed4cd5e41,client_4e07408562,1,0.496667,3283,0
3285,content_b6bb672d24a4,client_4e07408562,1,0.496667,3286,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*Weak picks are high-ranked items whose score may be driven by unstable or incomplete signals. In particular, percentage-free percentile scores can still prioritize pages with limited context, while seasonality or measurement changes may make a page look like a refresh opportunity. I will treat these as review candidates rather than confirmed actions. The baseline does not use trend_direction or trend_pct as scoring features, and it does not use future windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Find potentially weak high-ranked picks
weak_picks = output[
    (output["baseline_rank"] <= 20) &
    (
        (output["impressions_90d"] < 100) |
        (output["sessions_90d"] < 10) |
        (output["word_count"] <= 0)
    )
]

print("Potential weak picks in Top-20:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "baseline_rank",
                "content_id",
                "baseline_refresh_score",
                "suggested_action",
                "reason_codes",
                "impressions_90d",
                "sessions_90d",
                "word_count"
            ]
        ]
    )


# Leakage check
score_features = [
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score"
]

forbidden_fields = [
    "trend_direction",
    "trend_pct"
]

print("\nLeakage check")
print("Score features:", score_features)
print(
    "Forbidden label fields used:",
    [c for c in forbidden_fields if c in score_features]
)
print("Future-window fields used: none")

Potential weak picks in Top-20: 0

Leakage check
Score features: ['visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score']
Forbidden label fields used: []
Future-window fields used: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.